In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import re
import json
import numpy as np

In [4]:
#d = ['theme', 'nom_event', 'adresse', 'ville', 'cp', 'dep','adresse_complete', 'date_d', 'date_f', 'url_event', 'url_image', 'age', 'prix', 'keyword', 'desc', 'coor']

In [2]:
HEADERS = {"User-Agent": "Mozilla/5.0 Chrome/124.0"}

# FLANER BOUGER

In [ ]:
import requests
from bs4 import BeautifulSoup
import re
import time
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

session = requests.Session()
session.headers.update(HEADERS)

# ============== FONCTIONS UTILITAIRES ==============

def clean_text(text):
    if text is None:
        return None
    return re.sub(r"\s+", " ", text).strip()

def split_lieu(lieu_brut):
    if not lieu_brut:
        return None, None
    parts = lieu_brut.split(" - ", 1)
    ville = parts[0].strip() if len(parts) > 0 else None
    adresse = parts[1].strip() if len(parts) > 1 else None
    return ville, adresse

def parse_dates(raw):
    if not raw:
        return None, None
    raw = clean_text(raw)
    m = re.match(r"Du\s+(.+?)\s+au\s+(.+)", raw, re.IGNORECASE)
    if m:
        return m.group(1).strip(), m.group(2).strip()
    m2 = re.match(r"Le\s+(.+)", raw, re.IGNORECASE)
    if m2:
        return m2.group(1).strip(), None
    return raw, None

def get_event_details(url, session):
    try:
        r = session.get(url, timeout=10)
        soup = BeautifulSoup(r.text, "html.parser")
    except requests.RequestException:
        return None, None, None, None

    # Date
    date_raw = None
    for div in soup.find_all("div", class_="field-item"):
        if "Date :" in div.get_text():
            b = div.find("b")
            if b:
                date_raw = clean_text(b.get_text(" ", strip=True))
                break
    date_debut, date_fin = parse_dates(date_raw)

    # Website
    site_web = None
    for div in soup.find_all("div", class_="titreb"):
        if "Website" in div.get_text():
            next_div = div.find_next_sibling("div", class_="field-items")
            if next_div:
                a = next_div.find("a")
                if a:
                    site_web = a.get("href")
            break

    # Code postal — prend le 2ème s'il y en a deux
    cp = None
    for div in soup.find_all("div", class_="titreb"):
        if "Adresse" in div.get_text():
            next_div = div.find_next_sibling("div", class_="field-items")
            if next_div:
                texte = next_div.get_text(" ", strip=True)
                tous_les_cp = re.findall(r"\b\d{5}\b", texte)
                if len(tous_les_cp) >= 2:
                    cp = tous_les_cp[1]
                elif len(tous_les_cp) == 1:
                    cp = tous_les_cp[0]
            break

    return date_debut, date_fin, site_web, cp

def scrape_row(row, numero):
    title_div = row.find("div", class_="views-field-title")
    if not title_div:
        return None
    a_titre = title_div.find("a")
    titre = clean_text(a_titre.get_text(" ", strip=True)) if a_titre else None
    lien_relatif = a_titre.get("href") if a_titre else None
    lien = f"https://flanerbouger.fr{lien_relatif}" if lien_relatif and lien_relatif.startswith("/") else lien_relatif

    categorie, lieu = None, None
    info_divs = row.find_all("div", class_="views-field-field-location-taxonomize-terms-location-taxonomize-longname")
    for div in info_divs:
        label_span = div.find("span", class_="field-content")
        label = label_span.get_text(strip=True) if label_span else ""
        if "Catégorie" in label:
            cat_link = div.find("span", class_="views-field-field-tags")
            if cat_link:
                a_cat = cat_link.find("a")
                categorie = clean_text(a_cat.get_text(" ", strip=True)) if a_cat else None
        elif "Lieu" in label:
            value_span = div.find("span", class_="views-field-field-postaladdress-postal-code")
            if value_span:
                value_content = value_span.find("span", class_="field-content")
                lieu = clean_text(value_content.get_text(" ", strip=True)) if value_content else None

    ville_evt, adresse_evt = split_lieu(lieu)

    date_debut, date_fin, site_web, cp = None, None, None, None
    if lien:
        date_debut, date_fin, site_web, cp = get_event_details(lien, session)

    return {
        "theme": categorie,
        "nom_event": titre,
        "adresse": adresse_evt,
        "ville": ville_evt,
        "cp": cp,
        "dep": numero,
        "date_d": date_debut,
        "date_f": date_fin,
        "site_web": site_web,
    }

# ============== SCRAPING PRINCIPAL ==============

all_data = []

departements = {
    "01":"ain","02":"aisne","03":"allier","04":"alpes-de-haute-provence",
    "05":"hautes-alpes","06":"alpes-maritimes","07":"ardeche","08":"ardennes",
    "09":"ariege","10":"aube","11":"aude","12":"aveyron","13":"bouches-du-rhone",
    "14":"calvados","15":"cantal","16":"charente","17":"charente-maritime",
    "18":"cher","19":"correze","2A":"corse-du-Sud","2B":"haute-Corse",
    "21":"cote-d-or","22":"Cotes-d-armor","23":"creuse","24":"dordogne",
    "25":"doubs","26":"drome","27":"eure","28":"eure-et-loir","29":"finistere",
    "30":"gard","31":"haute-garonne","32":"gers","33":"gironde","34":"herault",
    "35":"ille-et-vilaine","36":"indre","37":"indre-et-loire","38":"isere",
    "39":"jura","40":"landes","41":"loir-et-cher","42":"loire","43":"haute-loire",
    "44":"loire-atlantique","45":"loiret","46":"lot","47":"lot-et-garonne",
    "48":"lozere","49":"maine-et-loire","50":"manche","51":"marne",
    "52":"haute-marne","53":"mayenne","54":"meurthe-et-moselle","55":"meuse",
    "56":"morbihan","57":"moselle","58":"nievre","59":"nord","60":"oise",
    "61":"orne","62":"pas-de-calais","63":"puy-de-dome","64":"Pyrenees-Atlantiques",
    "65":"hautes-pyrenees","66":"pyrenees-orientales","67":"bas-rhin","68":"haut-rhin",
    "69":"rhone","70":"haute-saone","71":"saone-et-loire","72":"Sarthe",
    "73":"savoie","74":"haute-savoie","75":"ile-de-france","76":"seine-maritime",
    "77":"seine-et-marne","78":"yvelines","79":"deux-sevres","80":"somme",
    "81":"tarn","82":"tarn-et-garonne","83":"var","84":"vaucluse","85":"vendee",
    "86":"vienne","87":"haute-vienne","88":"vosges","89":"yonne",
    "90":"territoire-de-belfort","91":"essonne","92":"hauts-de-seine",
    "93":"seine-saint-denis","94":"val-de-marne","95":"val-d-oise"
}

for numero, nom in departements.items():
    base_url = f"https://flanerbouger.fr/event/{numero}-event-{nom}"
    print(f"\n--- Département {numero} : {nom} ---")

    try:
        r = session.get(base_url, timeout=10)
        soup = BeautifulSoup(r.text, "html.parser")
    except Exception as e:
        print(f"Erreur sur {base_url} : {e}")
        continue

    page_numbers = [int(m.group(1)) for a in soup.find_all("a", href=True)
                    if (m := re.search(r"[?&]page=(\d+)", a["href"]))]
    last_page = max(page_numbers) if page_numbers else 1
    print(f"Pages détectées : {last_page}")

    for page in range(1, last_page + 1):
        url_page = base_url if page == 1 else f"{base_url}?page={page}"

        try:
            r = session.get(url_page, timeout=10)
            soup = BeautifulSoup(r.text, "html.parser")
        except Exception as e:
            print(f"Erreur page {page} : {e}")
            continue

        rows = soup.find_all("div", class_="views-row")
        if not rows:
            print(f"Page {page} vide, arrêt.")
            break

        page_data = []
        with ThreadPoolExecutor(max_workers=10) as executor:
            futures = {executor.submit(scrape_row, row, numero): row for row in rows}
            for future in as_completed(futures):
                try:
                    result = future.result()
                    if result:
                        page_data.append(result)
                except Exception as e:
                    print(f"Erreur sur un événement : {e}")

        print(f"  Page {page} → {len(page_data)} événements")
        all_data.extend(page_data)
        time.sleep(0.5)





--- Département 01 : ain ---
Pages détectées : 1
Page 1 vide, arrêt.

--- Département 02 : aisne ---
Pages détectées : 1
Page 1 vide, arrêt.

--- Département 03 : allier ---
Pages détectées : 1
Page 1 vide, arrêt.

--- Département 04 : alpes-de-haute-provence ---
Pages détectées : 1
Page 1 vide, arrêt.

--- Département 05 : hautes-alpes ---
Pages détectées : 1
Page 1 vide, arrêt.

--- Département 06 : alpes-maritimes ---
Pages détectées : 1
Page 1 vide, arrêt.

--- Département 07 : ardeche ---
Pages détectées : 1
Page 1 vide, arrêt.

--- Département 08 : ardennes ---
Pages détectées : 1
Page 1 vide, arrêt.

--- Département 09 : ariege ---
Pages détectées : 1
Page 1 vide, arrêt.

--- Département 10 : aube ---
Pages détectées : 1
Page 1 vide, arrêt.

--- Département 11 : aude ---
Pages détectées : 1
Page 1 vide, arrêt.

--- Département 12 : aveyron ---
Pages détectées : 1
Page 1 vide, arrêt.

--- Département 13 : bouches-du-rhone ---
Pages détectées : 1
Page 1 vide, arrêt.

--- Départem

: 

In [7]:
# ============== EXPORT ==============

df_flanerbouger = pd.DataFrame(all_data)

In [8]:
condition = df_flanerbouger["dep"] == "01"
df_flanerbouger[condition]

,theme,nom_event,adresse,ville,cp,dep,date_d,date_f,site_web
0,Marchés,Marché alimentaire vendredi de chaque semaine,"quartier gare, devant la gare sncf",AMBERIEU-EN-BUGEY,01500,01,03 July 2026,NaN,None
1,Brocantes,Broc'en Bresse,Halle du champ de foire,BOURG-EN-BRESSE,01000,01,03 July 2026,NaN,None
2,Marchés,Marché Hebdomadaire vendredi de chaque semaine,place Allombert,CERDON,01450,01,03 July 2026,NaN,None
3,Marchés,Marché hebdomadaire vendredi,centre,CHALAMONT,01320,01,03 July 2026,NaN,None
4,Marchés,Corveissiat fait son marché,Place Charles Blétel,CORVEISSIAT,01250,01,03 July 2026,NaN,None
...,...,...,...,...,...,...,...,...,...
289,Vide-greniers,Vide-greniers et marché aux puces,NaN,MONTAGNIEU -,01470,01,01 May 2027,NaN,None
290,Loisirs et tourisme,Chateau de voltaire a ferney,NaN,FERNEY-VOLTAIRE -,01210,01,31 December 2027,NaN,None
291,Concerts,Goldmen - concert best of goldman,NaN,BOURG-EN-BRESSE -,01000,01,17 November 2028,NaN,None
292,Loisirs et tourisme,Monastere royal de brou,NaN,BOURG-EN-BRESSE -,01000,01,31 December 2027,NaN,None


In [9]:
df_flanerbouger.info()

<class 'pandas.DataFrame'>
RangeIndex: 38611 entries, 0 to 38610
Data columns (total 9 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   theme      38611 non-null  str   
 1   nom_event  38611 non-null  str   
 2   adresse    19347 non-null  str   
 3   ville      38611 non-null  str   
 4   cp         38610 non-null  str   
 5   dep        38611 non-null  str   
 6   date_d     38610 non-null  str   
 7   date_f     639 non-null    str   
 8   site_web   0 non-null      object
dtypes: object(1), str(8)
memory usage: 5.7+ MB


In [10]:
print(df_flanerbouger['date_d'].isna().sum())
print(df_flanerbouger['date_d'].dropna().unique()[:5])

1
<ArrowStringArray>
['03 July 2026', '04 July 2026', '05 July 2026', '06 July 2026',
 '07 July 2026']
Length: 5, dtype: str


In [11]:
import locale
locale.setlocale(locale.LC_TIME, 'C')  # remet anglais

df_flanerbouger['date_d'] = pd.to_datetime(df_flanerbouger['date_d'].astype('object'), format='%d %B %Y', errors='coerce').dt.strftime('%Y-%m-%d')
df_flanerbouger['date_f'] = pd.to_datetime(df_flanerbouger['date_f'].astype('object'), format='%d %B %Y', errors='coerce').dt.strftime('%Y-%m-%d')

print(df_flanerbouger['date_d'].isna().sum())
print(df_flanerbouger['date_d'].dropna().unique()[:5])

31
<ArrowStringArray>
['2026-07-03', '2026-07-04', '2026-07-05', '2026-07-06', '2026-07-07']
Length: 5, dtype: str


In [12]:
df_flanerbouger['ville'] = df_flanerbouger['ville'].str.strip().str.rstrip('-').str.strip()

In [13]:
df_flanerbouger["adresse_complete"] = df_flanerbouger["adresse"].fillna("") + " " + df_flanerbouger["cp"].fillna("") + " " + df_flanerbouger["ville"].fillna("")

In [14]:
df_flanerbouger['adresse'] = df_flanerbouger['adresse'].str.replace(r'\s*\d{5}\s*-\s*.+$', '', regex=True).str.strip()

In [15]:
df_flanerbouger[['url_event', 'url_image', 'age', 'prix', 'keyword', 'desc', 'coor']] = np.nan

In [16]:
df_flanerbouger = df_flanerbouger[['theme', 'nom_event', 'adresse', 'ville', 'cp', 'dep', 'adresse_complete', 'date_d', 'date_f', 'url_event', 'url_image', 'age', 'prix', 'keyword', 'desc', 'coor']]

In [17]:
df_flanerbouger["date_f"] = df_flanerbouger["date_f"].fillna(df_flanerbouger["date_d"])

In [18]:
df_flanerbouger = df_flanerbouger.drop_duplicates(subset=["nom_event", "cp"], keep="first").drop_duplicates(subset=["nom_event", "cp"], keep="first")

In [19]:
df_flanerbouger = df_flanerbouger.dropna(subset="cp")

In [20]:
df_flanerbouger["theme"].unique()

<ArrowStringArray>
[                             'Marchés',
                            'Brocantes',
                 'Marchés de créateurs',
                              'Bourses',
               'Marchés de Producteurs',
                          'Marchés bio',
                        'Vide-greniers',
           'Manifestations Automobiles',
                        'Fêtes locales',
                               'Foires',
                               'Salons',
                            'Festivals',
                    'Marchés aux puces',
                               'Messes',
                      'Feux d'artifice',
                         'Sorties Moto',
                          'Bals Danses',
                             'Concerts',
 'Annonces cyclistes et cyclosportives',
                     'Fêtes médiévales',
                       'Fêtes foraines',
         'Randonnées, Trail ou Running',
                   'Marchés de potiers',
               'Journées du patrimoine

In [21]:
df_flanerbouger.info()

<class 'pandas.DataFrame'>
Index: 33490 entries, 0 to 38610
Data columns (total 16 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   theme             33490 non-null  str    
 1   nom_event         33490 non-null  str    
 2   adresse           17329 non-null  str    
 3   ville             33490 non-null  str    
 4   cp                33490 non-null  str    
 5   dep               33490 non-null  str    
 6   adresse_complete  33490 non-null  str    
 7   date_d            33461 non-null  str    
 8   date_f            33461 non-null  str    
 9   url_event         0 non-null      float64
 10  url_image         0 non-null      float64
 11  age               0 non-null      float64
 12  prix              0 non-null      float64
 13  keyword           0 non-null      float64
 14  desc              0 non-null      float64
 15  coor              0 non-null      float64
dtypes: float64(7), str(9)
memory usage: 8.2 MB


In [22]:
def getcoordinates(adresse):
    url = "https://data.geopf.fr/geocodage/search/"

    for _ in range(3):
        try:
            r = requests.get(
                url,
                params={"q": adresse},
                timeout=20
            )
            r.raise_for_status()

            data = r.json()
            return data["features"][0]["geometry"]["coordinates"][::-1]

        except requests.exceptions.Timeout:
            time.sleep(2)

        except (KeyError, IndexError):
            return None

        except requests.exceptions.RequestException:
            return None

    return None

In [23]:
from tqdm import tqdm

tqdm.pandas()  # active la méthode progress_apply sur les DataFrame/Series

df_flanerbouger["coor"] = df_flanerbouger["adresse_complete"].progress_apply(getcoordinates)

100%|██████████| 33490/33490 [4:12:41<00:00,  2.21it/s]   


In [24]:
df_flanerbouger.to_csv("df_flanerbouger.csv")

# PARC D'ATTRACTION

In [302]:
r = requests.get("https://www.parcs-france.com/carte/", headers=HEADERS)
soup = BeautifulSoup(r.text, "html.parser")

tbody = soup.find("tbody", {"class": "row-striping row-hover"})
rows = tbody.find_all("tr")

data = []

for row in rows:
    name_tag = row.find("a")  # peut être None

    name = name_tag.get_text(strip=True) if name_tag else row.get_text(strip=True)
    href = name_tag.get("href") if name_tag else None

    data.append({
        "name": name,
        "href": href
    })

df = pd.DataFrame(data)

In [303]:
def extract_info(url):

    if pd.isna(url):
        return {
            "adresse": None,
            "latitude": None,
            "longitude": None,
            "date_d": None,
            "date_f": None
        }

    r = requests.get(url, headers=HEADERS, timeout=15)
    soup = BeautifulSoup(r.text, "html.parser")

    adresse = None
    lat, lon = None, None
    date_d, date_f = None, None

    # Adresse
    for strong in soup.find_all("strong"):
        if "Adresse" in strong.get_text():
            next_node = strong.next_sibling
            if next_node:
                adresse = str(next_node).strip().lstrip(": ").strip()
            break

    # GPS
    gps_match = re.search(
        r"latitude\s*([-\d.]+)\s*\|\s*longitude\s*([-\d.]+)",
        soup.get_text(),
        re.IGNORECASE
    )

    if gps_match:
        lat = float(gps_match.group(1))
        lon = float(gps_match.group(2))

    # Dates d'ouverture
    text = soup.get_text(" ", strip=True)

    date_match = re.search(
        r"du\s+(\d{1,2}\s+\w+)\s+au\s+(\d{1,2}(?:er)?\s+\w+)\s+(\d{4})",
        text,
        re.IGNORECASE
    )

    if date_match:
        annee = date_match.group(3)
        date_d = f"{date_match.group(1)} {annee}"
        date_f = f"{date_match.group(2)} {annee}"

    return {
        "adresse": adresse,
        "latitude": lat,
        "longitude": lon,
        "date_d": date_d,
        "date_f": date_f
    }

In [304]:
df_info = df["href"].apply(extract_info).apply(pd.Series)

In [305]:
parc_attraction = pd.concat([df, df_info], axis=1)

In [306]:
parc_attraction.info()

<class 'pandas.DataFrame'>
RangeIndex: 33 entries, 0 to 32
Data columns (total 7 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   name       33 non-null     str    
 1   href       25 non-null     str    
 2   adresse    25 non-null     str    
 3   latitude   24 non-null     float64
 4   longitude  24 non-null     float64
 5   date_d     21 non-null     str    
 6   date_f     21 non-null     str    
dtypes: float64(2), str(5)
memory usage: 5.2 KB


In [307]:
parc_attraction["theme"] = "Parc d'attraction"

In [308]:
parc_attraction = parc_attraction.rename(columns={"name":"nom_event", "adresse":"adresse_complete"})

In [309]:
parc_attraction.sample(5)

,nom_event,href,adresse_complete,latitude,longitude,date_d,date_f,theme
18,OK Corral,https://www.parcs-france.com/ok-corral/,D8n 13780 Cuges-les-Pins,43.267338,5.727565,4 avril 2026,1er novembre 2026,Parc d'attraction
11,Kingoland,https://www.parcs-france.com/kingoland/,Pondigo – 56500 Plumelin,47.889351,-2.893787,11 avril 2026,27 septembre 2026,Parc d'attraction
20,Parc Astérix,https://www.parcs-france.com/parc-asterix/,BP8 – 60128 Plailly,49.135143,2.565823,3 octobre 2026,8 novembre 2026,Parc d'attraction
4,Didi'LandBas RhinGrand Est,NaN,NaN,NaN,NaN,NaN,NaN,Parc d'attraction
23,Parc du Petit Prince,https://www.parcs-france.com/parcdupetitprince/,rue de l’Espoir 68190 Ungersheim,47.863007,7.296180,3 avril 2026,1er novembre 2026,Parc d'attraction


In [69]:
# Extraction du code postal (5 chiffres)
parc_attraction["cp"] = parc_attraction["adresse_complete"].str.extract(r"\b(\d{5})\b")

# Extraction de la ville (tout ce qui suit le code postal)
parc_attraction["ville"] = parc_attraction["adresse_complete"].str.extract(r"\b\d{5}\s+(.+)$")

# Extraction de l'adresse (tout ce qui précède le code postal)
parc_attraction["adresse"] = parc_attraction["adresse_complete"].str.extract(r"^(.*?)\s+\d{5}\b").iloc[:, 0].str.strip()

In [70]:
parc_attraction.sample(10)

,nom_event,href,adresse_complete,latitude,longitude,date_d,date_f,theme,cp,ville,adresse
22,Parc du BocasseSeine MaritimeNormandie,NaN,NaN,NaN,NaN,NaN,NaN,Parc d'attraction,NaN,NaN,NaN
9,Futuroscope,https://www.parcs-france.com/futuroscope/,Avenue René Monory 86360 Chasseneuil-du-Poitou,46.663320,0.361748,NaN,NaN,Parc d'attraction,86360,Chasseneuil-du-Poitou,Avenue René Monory
13,La Récré des 3 Curés,https://www.parcs-france.com/larecredes3cures/,Les Trois Curés 29290 Milizac,48.475131,-4.526596,5 avril 2026,1er novembre 2026,Parc d'attraction,29290,Milizac,Les Trois Curés
7,FestylandCalvadosNormandie,NaN,NaN,NaN,NaN,NaN,NaN,Parc d'attraction,NaN,NaN,NaN
8,Fraispertuis City,https://www.parcs-france.com/fraispertuis-city/,50 rue de la colline des eaux 88700 Jeanménil,48.324940,6.726357,4 avril 2026,27 septembre 2026,Parc d'attraction,88700,Jeanménil,50 rue de la colline des eaux
26,Parc Saint Paul,https://www.parcs-france.com/parc-saint-paul/,RD931 60650 Saint-Paul,49.421169,1.982646,4 avril 2026,1er novembre 2026,Parc d'attraction,60650,Saint-Paul,RD931
19,Papéa Parc,https://www.parcs-france.com/papea-parc/,Lieu-dit Neptune 72530 Yvré-l’Évêque,48.002137,0.250825,4 avril 2026,27 septembre 2026,Parc d'attraction,72530,Yvré-l’Évêque,Lieu-dit Neptune
25,Parc Les NaudièresLoire AtlantiquePays de la L...,NaN,NaN,NaN,NaN,NaN,NaN,Parc d'attraction,NaN,NaN,NaN
5,Disneyland Paris,https://www.parcs-france.com/disneyland-paris/,Boulevard du Parc 77700 Serris/Coupvray,48.876077,2.796460,NaN,NaN,Parc d'attraction,77700,Serris/Coupvray,Boulevard du Parc
12,La Mer de Sable,https://www.parcs-france.com/mer-de-sable/,N330 – 60950 Ermenonville,49.146340,2.678949,11 avril 2026,1er novembre 2026,Parc d'attraction,60950,Ermenonville,N330 –


In [71]:
departements = {"01":"ain",
                "02":"aisne",
                "03":"allier",
                "04":"alpes-de-haute-provence",
                "05":"hautes-alpes",
                "06":"alpes-maritimes",
                "07":"ardeche",
                "08":"ardennes",
                "09":"ariege",
                "10":"aube",
                "11":"aude",
                "12":"aveyron",
                "13":"bouches-du-rhone",
                "14":"calvados",
                "15":"cantal",
                "16":"charente",
                "17":"charente-maritime",
                "18":"cher",
                "19":"correze",
                "2A":"corse-du-Sud",
                "2B":"haute-Corse",
                "21":"cote-d-or",
                "22":"Cotes-d-armor",
                "23":"creuse",
                "24":"dordogne",
                "25":"doubs",
                "26":"drome",
                "27":"eure",
                "28":"eure-et-loir",
                "29":"finistere",
                "30":"gard",
                "31":"haute-garonne",
                "32":"gers",
                "33":"gironde",
                "34":"herault",
                "35":"ille-et-vilaine",
                "36":"indre",
                "37":"indre-et-loire",
                "38":"isere",
                "39":"jura",
                "40":"landes",
                "41":"loir-et-cher",
                "42":"loire",
                "43" : "haute-loire",
                "44":"loire-atlantique",
                "45": "loiret",
                "46":"lot",
                "47":"lot-et-garonne",
                "48" : "lozere",
                "49":"maine-et-loire",
                "50":"manche",
                "51":"marne",
                "52":"haute-marne",
                "53":"mayenne",
                "54":"meurthe-et-moselle",
                "55":"meuse",
                "56":"morbihan","57":"moselle","58":"nievre","59":"nord","60":"oise","61":"orne","62":"pas-de-calais","63":"puy-de-dome", "64" : "Pyrenees-Atlantiques", "65":"hautes-pyrenees","66":"pyrenees-orientales","67":"bas-rhin","68":"haut-rhin","69":"rhone","70":"haute-saone","71":"saone-et-loire", "72" : "Sarthe", "73":"savoie","74":"haute-savoie","75":"ile-de-france","76":"seine-maritime","77":"seine-et-marne","78":"yvelines",
"79":"deux-sevres",
"80":"somme",
"81":"tarn",
"82":"tarn-et-garonne",
"83":"var",
"84":"vaucluse",
"85":"vendee",
"86":"vienne",
"87":"haute-vienne",
"88":"vosges",
"89":"yonne",
"90":"territoire-de-belfort",
"91":"essonne",
"92":"hauts-de-seine",
"93":"seine-saint-denis",
"94":"val-de-marne",
"95":"val-d-oise"}

parc_attraction["dep"] = (parc_attraction["cp"].astype(str).str[:2].map(departements))

In [72]:
colonnes = [[ 'url_event', 'url_image', 'age', 'prix', 'keyword', 'desc']]

In [73]:
for col in colonnes:
    parc_attraction[col] = np.nan

In [74]:
parc_attraction["coor"] = parc_attraction[["latitude","longitude"]].values.tolist()

In [75]:
parc_attraction = parc_attraction[['theme', 'nom_event', 'adresse', 'ville', 'cp', 'dep','adresse_complete', 'date_d', 'date_f', 'url_event', 'url_image', 'age', 'prix', 'keyword', 'desc', 'coor', 'latitude', 'longitude',]]

In [76]:
parc_attraction = parc_attraction.drop(columns=["latitude","longitude"])

In [140]:
parc_attraction[["date_d","date_f"]] = parc_attraction[["date_d","date_f"]].fillna("Se renseigner")

In [77]:
parc_attraction.info()

<class 'pandas.DataFrame'>
RangeIndex: 33 entries, 0 to 32
Data columns (total 16 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   theme             33 non-null     str    
 1   nom_event         33 non-null     str    
 2   adresse           24 non-null     str    
 3   ville             24 non-null     str    
 4   cp                24 non-null     str    
 5   dep               24 non-null     str    
 6   adresse_complete  25 non-null     str    
 7   date_d            21 non-null     str    
 8   date_f            21 non-null     str    
 9   url_event         0 non-null      float64
 10  url_image         0 non-null      float64
 11  age               0 non-null      float64
 12  prix              0 non-null      float64
 13  keyword           0 non-null      float64
 14  desc              0 non-null      float64
 15  coor              33 non-null     object 
dtypes: float64(6), object(1), str(9)
memory usage: 8.1+ KB


# PARC AQUATIQUE

In [120]:
def get_parcaqua_links():
    r = requests.get("https://www.parcs-aquatiques.com/parcs-aquatiques-france/", headers=HEADERS)
    soup = BeautifulSoup(r.text, "html.parser")
    parcaqua = {}
    for li in soup.select("ul li a[href*='parcs-aquatiques.com']"):
        name = li.get_text(strip=True)
        href = li["href"]
        if name and "/carte" not in href:
            parcaqua[name] = href
    return parcaqua

In [121]:
def extract_info(url):
    r = requests.get(url, headers=HEADERS, timeout=15)
    soup = BeautifulSoup(r.text, "html.parser")

    adresse = None
    lat, lon = None, None

    # Chercher les balises <strong> qui contiennent "Adresse"
    for strong in soup.find_all("strong"):
        if "Adresse" in strong.get_text():
            # Le texte de l'adresse est dans le noeud suivant (NavigableString)
            next_node = strong.next_sibling
            if next_node:
                adresse = str(next_node).strip().lstrip(": ").strip()
            break

    # Coordonnées GPS
    gps_match = re.search(r"latitude\s*([\d.]+)\s*\|\s*longitude\s*([\d.]+)", soup.get_text())
    if gps_match:
        lat = float(gps_match.group(1))
        lon = float(gps_match.group(2))

    return {"adresse": adresse, "latitude": lat, "longitude": lon}

parc_aquatique = get_parcaqua_links()
records = []
for name, url in parc_aquatique.items():
    print(f"→ {name}")
    info = extract_info(url)
    records.append({"nom": name, "url": url, **info})
    time.sleep(0.8)

df_aqua= pd.DataFrame(records)


→ Accueil
→ Promos
→ Carte France
→ Aqualand
→ Agen
→ Bassin d’Arcachon
→ Cap d’Agde
→ Fréjus
→ Port Leucate
→ Saint Cyprien
→ Saint Cyr sur Mer
→ Sainte Maxime
→ Aquasplash
→ Aquascope
→ Aquaboulevard
→ Aqua Béarnà Oloron Sainte Marie
→ Aquaboulevardà Paris
→ Aqua’Fun Park – Cobac Parcà Lanhélin
→ Aquajetà Narbonne Plage
→ Aqualand Agen– Walygator Sud Ouest
→ Aqualand Bassin d’Arcachon
→ Aqualand Fréjus
→ Aqualand Le Cap d’Agde
→ Aqualand Port Leucate
→ Aqualand Saint Cyprien
→ Aqualand Saint Cyr sur Mer
→ Aqualand Sainte Maxime
→ Aquaparc Isisà Dole
→ Aquascope Futuroscope
→ Aquasplashà Antibes
→ Aquatic Landesà Labenne-Océan
→ Atlantic Parkà Seignosse Océan
→ Atlantic Tobogganà Saint-Hilaire-de-Riez
→ Espace Grand Bleuà La Grande Motte
→ Iléosur l’île d’Oléron
→ Ludolacà Vesoul
→ Nyonsoleïadoà Nyons
→ O’Gliss Parken Vendée
→ Parc de la Bouscarasseà Serviers-et-Labaume
→ Pirates WorldCap d’Agde
→ Quercylandà Souillac
→ Vitam à Neydens
→ Wave Islandà Monteux
→ Western Parkà Biguglia
→

In [122]:
df_aqua = df_aqua.dropna(subset="adresse")

In [123]:
df_aqua["theme"] = "Parc aquatique"

In [124]:
df_aqua = df_aqua.rename(columns={"nom":"nom_event", "adresse":"adresse_complete"})

In [125]:
# Extraction du code postal (5 chiffres)
df_aqua["cp"] = df_aqua["adresse_complete"].str.extract(r"\b(\d{5})\b")

# Extraction de la ville (tout ce qui suit le code postal)
df_aqua["ville"] = df_aqua["adresse_complete"].str.extract(r"\b\d{5}\s+(.+)$")

# Extraction de l'adresse (tout ce qui précède le code postal)
df_aqua["adresse"] = df_aqua["adresse_complete"].str.extract(r"^(.*?)\s+\d{5}\b").iloc[:, 0].str.strip()

In [126]:
departements = {"01":"ain",
                "02":"aisne",
                "03":"allier",
                "04":"alpes-de-haute-provence",
                "05":"hautes-alpes",
                "06":"alpes-maritimes",
                "07":"ardeche",
                "08":"ardennes",
                "09":"ariege",
                "10":"aube",
                "11":"aude",
                "12":"aveyron",
                "13":"bouches-du-rhone",
                "14":"calvados",
                "15":"cantal",
                "16":"charente",
                "17":"charente-maritime",
                "18":"cher",
                "19":"correze",
                "2A":"corse-du-Sud",
                "2B":"haute-Corse",
                "21":"cote-d-or",
                "22":"Cotes-d-armor",
                "23":"creuse",
                "24":"dordogne",
                "25":"doubs",
                "26":"drome",
                "27":"eure",
                "28":"eure-et-loir",
                "29":"finistere",
                "30":"gard",
                "31":"haute-garonne",
                "32":"gers",
                "33":"gironde",
                "34":"herault",
                "35":"ille-et-vilaine",
                "36":"indre",
                "37":"indre-et-loire",
                "38":"isere",
                "39":"jura",
                "40":"landes",
                "41":"loir-et-cher",
                "42":"loire",
                "43" : "haute-loire",
                "44":"loire-atlantique",
                "45": "loiret",
                "46":"lot",
                "47":"lot-et-garonne",
                "48" : "lozere",
                "49":"maine-et-loire",
                "50":"manche",
                "51":"marne",
                "52":"haute-marne",
                "53":"mayenne",
                "54":"meurthe-et-moselle",
                "55":"meuse",
                "56":"morbihan","57":"moselle","58":"nievre","59":"nord","60":"oise","61":"orne","62":"pas-de-calais","63":"puy-de-dome", "64" : "Pyrenees-Atlantiques", "65":"hautes-pyrenees","66":"pyrenees-orientales","67":"bas-rhin","68":"haut-rhin","69":"rhone","70":"haute-saone","71":"saone-et-loire", "72" : "Sarthe", "73":"savoie","74":"haute-savoie","75":"ile-de-france","76":"seine-maritime","77":"seine-et-marne","78":"yvelines",
"79":"deux-sevres",
"80":"somme",
"81":"tarn",
"82":"tarn-et-garonne",
"83":"var",
"84":"vaucluse",
"85":"vendee",
"86":"vienne",
"87":"haute-vienne",
"88":"vosges",
"89":"yonne",
"90":"territoire-de-belfort",
"91":"essonne",
"92":"hauts-de-seine",
"93":"seine-saint-denis",
"94":"val-de-marne",
"95":"val-d-oise"}

df_aqua["dep"] = (df_aqua["cp"].astype(str).str[:2].map(departements))

In [127]:
colonnes = [[ 'url_event', 'url_image', 'age', 'prix', 'keyword', 'desc', "date_d","date_f"]]

In [128]:
for col in colonnes:
    df_aqua[col] = np.nan

In [129]:
df_aqua["coor"] = df_aqua[["latitude","longitude"]].values.tolist()

In [130]:
df_aqua = df_aqua[['theme', 'nom_event', 'adresse', 'ville', 'cp', 'dep','adresse_complete', 'date_d', 'date_f', 'url_event', 'url_image', 'age', 'prix', 'keyword', 'desc', 'coor', 'latitude', 'longitude',]]

In [131]:
df_aqua = df_aqua.drop(columns=["latitude","longitude"])

In [132]:
df_aqua[["date_d","date_f"]] = df_aqua[["date_d","date_f"]].fillna("Se renseigner")

In [139]:
df_aqua.sample(7)

,theme,nom_event,adresse,ville,cp,dep,adresse_complete,date_d,date_f,url_event,url_image,age,prix,keyword,desc,coor
21,Parc aquatique,Aqualand Fréjus,Quartier le Capou D559,Fréjus,83600,var,Quartier le Capou D559 83600 Fréjus,Se renseigner,Se renseigner,NaN,NaN,NaN,NaN,NaN,NaN,"[43.419434, 6.728501]"
6,Parc aquatique,Cap d’Agde,2 avenue des Iles d’Amérique,Le Cap d’Agde,34300,herault,2 avenue des Iles d’Amérique 34300 Le Cap d’Agde,Se renseigner,Se renseigner,NaN,NaN,NaN,NaN,NaN,NaN,"[43.280843, 3.494517]"
47,Parc aquatique,• Plopsaqua,NaN,NaN,NaN,NaN,De Pannelaan 68 B-8660 De Panne (Belgique),Se renseigner,Se renseigner,NaN,NaN,NaN,NaN,NaN,NaN,"[51.081758, 2.601974]"
26,Parc aquatique,Aqualand Sainte Maxime,Avenue Gaston Rebuffat,Sainte Maxime,83120,var,Avenue Gaston Rebuffat 83120 Sainte Maxime,Se renseigner,Se renseigner,NaN,NaN,NaN,NaN,NaN,NaN,"[43.327553, 6.619147]"
16,Parc aquatique,Aquaboulevardà Paris,4-6 rue Louis Armand,Paris,75015,ile-de-france,4-6 rue Louis Armand 75015 Paris,Se renseigner,Se renseigner,NaN,NaN,NaN,NaN,NaN,NaN,"[48.833049, 2.277545]"
18,Parc aquatique,Aquajetà Narbonne Plage,route de Gruissan,Narbonne Plage,11100,aude,route de Gruissan 11100 Narbonne Plage,Se renseigner,Se renseigner,NaN,NaN,NaN,NaN,NaN,NaN,"[43.147748, 3.1531896]"
20,Parc aquatique,Aqualand Bassin d’Arcachon,route des lacs,Gujan-Mestras,33470,gironde,route des lacs 33470 Gujan-Mestras,Se renseigner,Se renseigner,NaN,NaN,NaN,NaN,NaN,NaN,"[nan, nan]"


# ZOO

In [176]:
def get_zoo_links():
    r = requests.get("https://www.zoofrance.com/carte/", headers=HEADERS)
    soup = BeautifulSoup(r.text, "html.parser")
    zoos = {}
    for li in soup.select("ul li a[href*='zoofrance.com']"):
        name = li.get_text(strip=True)
        href = li["href"]
        if name and "/carte" not in href:
            zoos[name] = href
    return zoos

def extract_info(url):
    r = requests.get(url, headers=HEADERS, timeout=15)
    soup = BeautifulSoup(r.text, "html.parser")

    adresse = None
    lat, lon = None, None

    # Chercher les balises <strong> qui contiennent "Adresse"
    for strong in soup.find_all("strong"):
        if "Adresse" in strong.get_text():
            # Le texte de l'adresse est dans le noeud suivant (NavigableString)
            next_node = strong.next_sibling
            if next_node:
                adresse = str(next_node).strip().lstrip(": ").strip()
            break

    # Coordonnées GPS
    gps_match = re.search(r"latitude\s*([\d.]+)\s*\|\s*longitude\s*([\d.]+)", soup.get_text())
    if gps_match:
        lat = float(gps_match.group(1))
        lon = float(gps_match.group(2))

    return {"adresse": adresse, "latitude": lat, "longitude": lon}

zoos = get_zoo_links()
records = []
for name, url in zoos.items():
    info = extract_info(url)
    records.append({"nom": name, "url": url, **info})
    time.sleep(0.8)

df_zoo = pd.DataFrame(records)


In [177]:
df_zoo = df.dropna(subset="adresse")

In [178]:
df_zoo["theme"] = "Zoo"

In [179]:
df_zoo = df_zoo.rename(columns={"nom":"nom_event"})

In [180]:
df_zoo.head()

,theme,nom_event,adresse,ville,cp,dep,adresse_complete,date_d,date_f,url_event,url_image,age,prix,keyword,desc,coor
4,Zoo,Agen,Château de Caudouin,Roquefort,47310,lot-et-garonne,Château de Caudouin 47310 Roquefort,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"[44.186173, 0.579221]"
5,Zoo,Bassin d’Arcachon,route des lacs,Gujan-Mestras,33470,gironde,route des lacs 33470 Gujan-Mestras,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"[nan, nan]"
6,Zoo,Cap d’Agde,2 avenue des Iles d’Amérique,Le Cap d’Agde,34300,herault,2 avenue des Iles d’Amérique 34300 Le Cap d’Agde,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"[43.280843, 3.494517]"
7,Zoo,Fréjus,Quartier le Capou D559,Fréjus,83600,var,Quartier le Capou D559 83600 Fréjus,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"[43.419434, 6.728501]"
8,Zoo,Port Leucate,avenue du Roussillon,Port Leucate,11370,aude,avenue du Roussillon 11370 Port Leucate,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"[42.842203, 3.042208]"


In [181]:
print(df_zoo.columns[df_zoo.columns.duplicated()])  # affiche les doublons
df_zoo = df_zoo.loc[:, ~df_zoo.columns.duplicated()]  # supprime les doublons

Index([], dtype='str')


In [182]:
departements = {"01":"ain",
                "02":"aisne",
                "03":"allier",
                "04":"alpes-de-haute-provence",
                "05":"hautes-alpes",
                "06":"alpes-maritimes",
                "07":"ardeche",
                "08":"ardennes",
                "09":"ariege",
                "10":"aube",
                "11":"aude",
                "12":"aveyron",
                "13":"bouches-du-rhone",
                "14":"calvados",
                "15":"cantal",
                "16":"charente",
                "17":"charente-maritime",
                "18":"cher",
                "19":"correze",
                "2A":"corse-du-Sud",
                "2B":"haute-Corse",
                "21":"cote-d-or",
                "22":"Cotes-d-armor",
                "23":"creuse",
                "24":"dordogne",
                "25":"doubs",
                "26":"drome",
                "27":"eure",
                "28":"eure-et-loir",
                "29":"finistere",
                "30":"gard",
                "31":"haute-garonne",
                "32":"gers",
                "33":"gironde",
                "34":"herault",
                "35":"ille-et-vilaine",
                "36":"indre",
                "37":"indre-et-loire",
                "38":"isere",
                "39":"jura",
                "40":"landes",
                "41":"loir-et-cher",
                "42":"loire",
                "43" : "haute-loire",
                "44":"loire-atlantique",
                "45": "loiret",
                "46":"lot",
                "47":"lot-et-garonne",
                "48" : "lozere",
                "49":"maine-et-loire",
                "50":"manche",
                "51":"marne",
                "52":"haute-marne",
                "53":"mayenne",
                "54":"meurthe-et-moselle",
                "55":"meuse",
                "56":"morbihan","57":"moselle","58":"nievre","59":"nord","60":"oise","61":"orne","62":"pas-de-calais","63":"puy-de-dome", "64" : "Pyrenees-Atlantiques", "65":"hautes-pyrenees","66":"pyrenees-orientales","67":"bas-rhin","68":"haut-rhin","69":"rhone","70":"haute-saone","71":"saone-et-loire", "72" : "Sarthe", "73":"savoie","74":"haute-savoie","75":"ile-de-france","76":"seine-maritime","77":"seine-et-marne","78":"yvelines",
"79":"deux-sevres",
"80":"somme",
"81":"tarn",
"82":"tarn-et-garonne",
"83":"var",
"84":"vaucluse",
"85":"vendee",
"86":"vienne",
"87":"haute-vienne",
"88":"vosges",
"89":"yonne",
"90":"territoire-de-belfort",
"91":"essonne",
"92":"hauts-de-seine",
"93":"seine-saint-denis",
"94":"val-de-marne",
"95":"val-d-oise"}

df_zoo["dep"] = (df_zoo["cp"].astype(str).str[:2].map(departements))

In [183]:
colonnes = [[ 'url_event', 'url_image', 'age', 'prix', 'keyword', 'desc', "date_d","date_f"]]

In [184]:
for col in colonnes:
    df_zoo[col] = np.nan

In [185]:
df_zoo = df_zoo[['theme', 'nom_event', 'adresse', 'ville', 'cp', 'dep','adresse_complete', 'date_d', 'date_f', 'url_event', 'url_image', 'age', 'prix', 'keyword', 'desc', 'coor',]]

In [186]:
import locale
locale.setlocale(locale.LC_TIME, 'French_France.1252')

# Recharge depuis la source avant toute conversion
df_zoo["date_d"] = pd.to_datetime(df_zoo["date_d"], errors='coerce').fillna(pd.Timestamp('2024-01-01'))
df_zoo["date_f"] = pd.to_datetime(df_zoo["date_f"], errors='coerce').fillna(pd.Timestamp('2027-12-31'))

df_zoo["date_d"] = df_zoo["date_d"].dt.strftime('%#d %B %Y')
df_zoo["date_f"] = df_zoo["date_f"].dt.strftime('%#d %B %Y')

In [187]:
df_zoo.head()

,theme,nom_event,adresse,ville,cp,dep,adresse_complete,date_d,date_f,url_event,url_image,age,prix,keyword,desc,coor
4,Zoo,Agen,Château de Caudouin,Roquefort,47310,lot-et-garonne,Château de Caudouin 47310 Roquefort,1 janvier 2024,31 décembre 2027,NaN,NaN,NaN,NaN,NaN,NaN,"[44.186173, 0.579221]"
5,Zoo,Bassin d’Arcachon,route des lacs,Gujan-Mestras,33470,gironde,route des lacs 33470 Gujan-Mestras,1 janvier 2024,31 décembre 2027,NaN,NaN,NaN,NaN,NaN,NaN,"[nan, nan]"
6,Zoo,Cap d’Agde,2 avenue des Iles d’Amérique,Le Cap d’Agde,34300,herault,2 avenue des Iles d’Amérique 34300 Le Cap d’Agde,1 janvier 2024,31 décembre 2027,NaN,NaN,NaN,NaN,NaN,NaN,"[43.280843, 3.494517]"
7,Zoo,Fréjus,Quartier le Capou D559,Fréjus,83600,var,Quartier le Capou D559 83600 Fréjus,1 janvier 2024,31 décembre 2027,NaN,NaN,NaN,NaN,NaN,NaN,"[43.419434, 6.728501]"
8,Zoo,Port Leucate,avenue du Roussillon,Port Leucate,11370,aude,avenue du Roussillon 11370 Port Leucate,1 janvier 2024,31 décembre 2027,NaN,NaN,NaN,NaN,NaN,NaN,"[42.842203, 3.042208]"


In [195]:
df_final_jade = pd.concat([df_flanerbouger,df_aqua,df_zoo,parc_attraction])

In [196]:
df_final_jade.info()

<class 'pandas.DataFrame'>
Index: 33657 entries, 0 to 32
Data columns (total 16 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   theme             33657 non-null  str    
 1   nom_event         33657 non-null  str    
 2   adresse           17435 non-null  str    
 3   ville             33647 non-null  str    
 4   cp                33647 non-null  str    
 5   dep               33645 non-null  str    
 6   adresse_complete  33649 non-null  str    
 7   date_d            33657 non-null  object 
 8   date_f            33657 non-null  object 
 9   url_event         0 non-null      float64
 10  url_image         0 non-null      float64
 11  age               0 non-null      float64
 12  prix              0 non-null      float64
 13  keyword           0 non-null      float64
 14  desc              0 non-null      float64
 15  coor              118 non-null    object 
dtypes: float64(6), object(3), str(7)
memory usage: 7.6+ MB


In [198]:
df_final_jade = df_final_jade.dropna(subset="adresse_complete")

In [199]:
df_final_jade.info()

<class 'pandas.DataFrame'>
Index: 33649 entries, 0 to 32
Data columns (total 16 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   theme             33649 non-null  str    
 1   nom_event         33649 non-null  str    
 2   adresse           17435 non-null  str    
 3   ville             33647 non-null  str    
 4   cp                33647 non-null  str    
 5   dep               33645 non-null  str    
 6   adresse_complete  33649 non-null  str    
 7   date_d            33649 non-null  object 
 8   date_f            33649 non-null  object 
 9   url_event         0 non-null      float64
 10  url_image         0 non-null      float64
 11  age               0 non-null      float64
 12  prix              0 non-null      float64
 13  keyword           0 non-null      float64
 14  desc              0 non-null      float64
 15  coor              110 non-null    object 
dtypes: float64(6), object(3), str(7)
memory usage: 7.6+ MB


In [203]:
df_final_jade['date_d'] = pd.to_datetime(df_final_jade['date_d'], format='mixed', dayfirst=True, errors='coerce')
df_final_jade['date_f'] = pd.to_datetime(df_final_jade['date_f'], format='mixed', dayfirst=True, errors='coerce')

df_final_jade['date_d'] = df_final_jade['date_d'].dt.strftime('%Y-%m-%d')
df_final_jade['date_f'] = df_final_jade['date_f'].dt.strftime('%Y-%m-%d')

In [204]:
mask = pd.to_datetime(df_final_jade['date_d'], format='mixed', dayfirst=True, errors='coerce').isna()
print(df_final_jade.loc[mask, 'date_d'].unique())

<ArrowStringArray>
[nan]
Length: 1, dtype: str


In [205]:
condition = df_final_jade["date_d"].isna()
df_final_jade[condition]

,theme,nom_event,adresse,ville,cp,dep,adresse_complete,date_d,date_f,url_event,url_image,age,prix,keyword,desc,coor
483,Fêtes médiévales,La Fête de la sorcière,"Château de Coucy Rue du Château, 02380 Coucy-l...",COUCY-LE-CHATEAU-AUFFRIQUE,02380,02,"Château de Coucy Rue du Château, 02380 Coucy-l...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1029,Fêtes locales,Fête de la transhumance,Place Marcel Sauvaire,CASTELLANE,04120,04,Place Marcel Sauvaire 04120 CASTELLANE,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1968,"Randonnées, Trail ou Running",RBMA... le retour (raid blanc de la Montagne A...,Station de la Croix de Bauzon,BORNE,07590,07,Station de la Croix de Bauzon 07590 BORNE,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1970,Foires,FOIRE DE LA SAINT-ANTOINE,"boulevards Pasteur, Gambetta et Vernon Aubenas...",AUBENAS,07200,07,"boulevards Pasteur, Gambetta et Vernon Aubenas...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2832,Stages Ateliers,stage de poterie : La vie en Relief,1285 Chemin du moulin du Fort,AIX-EN-PROVENCE,13540,13,1285 Chemin du moulin du Fort 13540 AIX-EN-PRO...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
27,Parc d'attraction,Parc Spirou,1 rue Jean-Henri Fabre,Monteux,84170,vaucluse,1 rue Jean-Henri Fabre 84170 Monteux,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"[44.0195121, 4.9619912]"
29,Parc d'attraction,Vulcania,2 Route de Mazayes,Saint Ours les Roches,63230,puy-de-dome,2 Route de Mazayes 63230 Saint Ours les Roches,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"[45.813872, 2.95247]"
30,Parc d'attraction,Walibi Rhône-Alpes,1380 route de la Corneille,Les Avenières-Veyrins-Thuellin,38630,isere,1380 route de la Corneille 38630 Les Avenières...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"[45.619205, 5.570697]"
31,Parc d'attraction,Walygator Grand Est,Voie Romaine,Maizières-lès-Metz,57280,moselle,Voie Romaine 57280 Maizières-lès-Metz,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"[49.225028, 6.155297]"
